<a href="https://colab.research.google.com/github/thabomosenthal/symmetrical-fortnight/blob/main/Week3_12_guardrails_safety.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.3: Guardrails - Safe AI Systems

## Key Insight: How to mitigate undesired outputs from LLMs

Make your digital twin SAFE for production!

## What You'll Build
- Content filters
- Output validators
- Safety layers
- Fallback strategies
- PII detection

In [1]:
import re
from typing import List, Dict, Any, Optional

print('Week 3.3: Guardrails & Safety')
print('Goal: Make your twin safe for real users!')

Week 3.3: Guardrails & Safety
Goal: Make your twin safe for real users!


## Part 1: Content Filtering

Block inappropriate content.

In [10]:
class ContentFilter:
    '''Filter inappropriate content with keyword-based topic blocking.'''

    def __init__(self):
        # Simplified word lists for demo
        self.profanity_list = ['badword1', 'badword2', 'shit', 'fuck', 'pussy']

        # Mapping sensitive topics to specific keywords for keyword-based blocking
        self.topic_keywords = {
            'violence': ['violence', 'fight', 'kill', 'attack', 'weapon'],
            'illegal': ['illegal', 'drugs', 'steal', 'hack', 'pirate','kidnap'],
            'sexuality': ['sexuality', 'porn', 'nsfw', 'adult content', 'gay']
        }

    def check_profanity(self, text: str) -> Dict[str, Any]:
        '''Check for profanity.'''
        text_lower = text.lower()
        found = [word for word in self.profanity_list if word in text_lower]

        return {
            'is_safe': len(found) == 0,
            'violations': found
        }

    def check_sensitive_topics(self, text: str) -> Dict[str, Any]:
        '''Check for sensitive topics based on keywords.'''
        text_lower = text.lower()
        flagged_topics = []

        for topic, keywords in self.topic_keywords.items():
            found_keywords = [kw for kw in keywords if kw in text_lower]
            if found_keywords:
                flagged_topics.append(topic)

        return {
            'is_safe': len(flagged_topics) == 0,
            'topics': flagged_topics
        }

    def filter(self, text: str) -> Dict[str, Any]:
        '''Run all filters.'''
        profanity_check = self.check_profanity(text)
        topic_check = self.check_sensitive_topics(text)

        is_safe = profanity_check['is_safe'] and topic_check['is_safe']

        return {
            'is_safe': is_safe,
            'profanity_violations': profanity_check['violations'],
            'sensitive_topics': topic_check['topics'],
            'text': text if is_safe else '[FILTERED]'
        }

filter_system = ContentFilter()

print('Content Filter Tests:')
print(filter_system.filter('Hello, how are you?'))
print(filter_system.filter('This involves a fight with weapons.'))  # Will flag 'violence'
print(filter_system.filter('What the fuck is going on here?')) # Should be flagged


Content Filter Tests:
{'is_safe': True, 'profanity_violations': [], 'sensitive_topics': [], 'text': 'Hello, how are you?'}
{'is_safe': False, 'profanity_violations': [], 'sensitive_topics': ['violence'], 'text': '[FILTERED]'}
{'is_safe': False, 'profanity_violations': ['fuck'], 'sensitive_topics': [], 'text': '[FILTERED]'}


## Part 2: PII Detection

Detect and protect personal information.

In [15]:
class PIIDetector:
    '''Detect personally identifiable information with precise patterns and context checks to minimize false positives.'''

    def __init__(self):
        # Regex patterns for common PII
        self.patterns = {
            'email': r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b',
            'phone': r'\b(?:\+?1[-. ]?)?(?:\([0-9]{3}\)|[0-9]{3})[-. ]?[0-9]{3}[-. ]?[0-9]{4}\b',
            'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
            'credit_card': r'\b(?:\d{4}[ -]?){3}\d{4}\b'
        }
        # Context keywords to confirm a matched sequence is a phone number
        self.phone_context_keywords = ['call', 'phone', 'ph', 'cell', 'tel', 'contact', 'mobile']

    def _has_phone_context(self, text: str, start_idx: int, end_idx: int) -> bool:
        '''Check if there is phone context around the matched string location.'''
        window_size = 25  # Look up to 25 characters before or after
        left_boundary = max(0, start_idx - window_size)
        right_boundary = min(len(text), end_idx + window_size)

        context_window = (text[left_boundary:start_idx] + " " + text[end_idx:right_boundary]).lower()
        return any(keyword in context_window for keyword in self.phone_context_keywords)

    def detect(self, text: str) -> Dict[str, Any]:
        '''Detect PII in text using pattern matching and context confirmation.'''
        found_pii = {}

        for pii_type, pattern in self.patterns.items():
            if pii_type == 'phone':
                # For phone numbers, find matches with positions to perform context checking
                phone_matches = []
                for match in re.finditer(pattern, text):
                    match_str = match.group()
                    start_idx, end_idx = match.span()
                    if self._has_phone_context(text, start_idx, end_idx):
                        phone_matches.append(match_str)
                if phone_matches:
                    found_pii['phone'] = phone_matches
            else:
                matches = re.findall(pattern, text)
                if matches:
                    found_pii[pii_type] = matches

        return {
            'has_pii': len(found_pii) > 0,
            'pii_types': list(found_pii.keys()),
            'details': found_pii
        }

    def redact(self, text: str) -> str:
        '''Redact PII from text safely using context rules where applicable.'''
        result = text

        # Redact non-phone patterns directly
        for pii_type, pattern in self.patterns.items():
            if pii_type == 'phone':
                # Reconstruct string by redacting only confirmed phone matches
                confirmed_matches = []
                for match in re.finditer(pattern, text):
                    if self._has_phone_context(text, match.start(), match.end()):
                        confirmed_matches.append((match.start(), match.end()))

                # Replace from right to left to maintain indices
                for start_idx, end_idx in sorted(confirmed_matches, reverse=True):
                    result = result[:start_idx] + '[REDACTED_PHONE]' + result[end_idx:]
            else:
                result = re.sub(pattern, f'[REDACTED_{pii_type.upper()}]', result)

        return result

pii_detector = PIIDetector()

# Let's test with valid PII and potential false-positives (like random numeric IDs)
text_with_pii = "Contact me at john@example.com or call (555) 123-4567. Do not flag ID: 1234567890 or ticket 123-45-67890."
print('\nPII Detection:')
print('Original:', text_with_pii)
print('Detection:', pii_detector.detect(text_with_pii))
print('Redacted:', pii_detector.redact(text_with_pii))


PII Detection:
Original: Contact me at john@example.com or call (555) 123-4567. Do not flag ID: 1234567890 or ticket 123-45-67890.
Detection: {'has_pii': True, 'pii_types': ['email'], 'details': {'email': ['john@example.com']}}
Redacted: Contact me at [REDACTED_EMAIL] or call (555) 123-4567. Do not flag ID: 1234567890 or ticket 123-45-67890.


## Part 3: Output Validator

Ensure AI outputs meet requirements.

In [18]:
class OutputValidator:
    '''Validate AI outputs before showing to user.'''

    def __init__(self, max_length: int = 500):
        self.max_length = max_length
        self.content_filter = ContentFilter()
        self.pii_detector = PIIDetector()
        # Compile regex patterns for disclaimer keywords on word boundaries
        self.disclaimer_patterns = {
            'medical': re.compile(r'\b(medical|doctor|medicine|health|pill|aspirin|treatment)\b', re.IGNORECASE),
            'legal': re.compile(r'\b(legal|lawyer|attorney|law|court|sue)\b', re.IGNORECASE),
            'financial': re.compile(r'\b(financial|finance|invest|stocks|advisor|money|portfolio)\b', re.IGNORECASE)
        }

    def validate(self, output: str) -> Dict[str, Any]:
        '''Run all validation checks.'''
        issues = []

        # Check length
        if len(output) > self.max_length:
            issues.append(f'Too long ({len(output)} > {self.max_length} chars)')

        # Check if empty
        if not output.strip():
            issues.append('Empty output')

        # Check content safety
        content_check = self.content_filter.filter(output)
        if not content_check['is_safe']:
            issues.append('Content safety violation')

        # Check for PII
        pii_check = self.pii_detector.detect(output)
        if pii_check['has_pii']:
            issues.append(f"Contains PII: {pii_check['pii_types']}")

        # Check for disclaimers using regex patterns
        needs_disclaimer = False
        matched_topics = []
        for topic, pattern in self.disclaimer_patterns.items():
            if pattern.search(output):
                needs_disclaimer = True
                matched_topics.append(topic)

        return {
            'is_valid': len(issues) == 0,
            'issues': issues,
            'needs_disclaimer': needs_disclaimer,
            'matched_topics': matched_topics,
            'output': output
        }

    def add_disclaimer(self, output: str, topic: str) -> str:
        '''Add disclaimer to output.'''
        disclaimers = {
            'medical': "\n\n⚠️ Note: This is not medical advice. Consult a healthcare professional.",
            'legal': "\n\n⚠️ Note: This is not legal advice. Consult a lawyer.",
            'financial': "\n\n⚠️ Note: This is not financial advice. Consult a financial advisor."
        }

        for key, disclaimer in disclaimers.items():
            if key in topic.lower():
                return output + disclaimer

        return output

validator = OutputValidator(max_length=300)

test_output = "You should take two tablets of aspirin if you feel a medical headache."
result = validator.validate(test_output)

print('\nValidation Result:')
print(f"Valid: {result['is_valid']}")
print(f"Issues: {result['issues']}")
print(f"Needs disclaimer: {result['needs_disclaimer']}")
print(f"Matched Topics: {result.get('matched_topics', [])}")

if result['needs_disclaimer']:
    # Use the first matched topic for applying correct disclaimer
    primary_topic = result.get('matched_topics', ['medical'])[0]
    safe_output = validator.add_disclaimer(test_output, primary_topic)
    print(f"\nWith disclaimer: {safe_output}")


Validation Result:
Valid: True
Issues: []
Needs disclaimer: True
Matched Topics: ['medical']

With disclaimer: You should take two tablets of aspirin if you feel a medical headache.

⚠️ Note: This is not medical advice. Consult a healthcare professional.


## Part 4: Safe Digital Twin

Put it all together.

In [19]:
class SafeDigitalTwin:
    '''Digital twin with safety guardrails.'''

    def __init__(self, name: str):
        self.name = name
        self.validator = OutputValidator()
        self.content_filter = ContentFilter()

    def _process_input(self, user_input: str) -> Optional[str]:
        '''Process and filter user input.'''
        # Filter input
        filter_result = self.content_filter.filter(user_input)

        if not filter_result['is_safe']:
            return None  # Input blocked

        return filter_result['text']

    def _generate_response(self, filtered_input: str) -> str:
        '''Generate AI response (mock for demo).'''
        # This would call your actual AI
        return f"I received: {filtered_input}"

    def _process_output(self, response: str) -> str:
        '''Validate and process output.'''
        validation = self.validator.validate(response)

        if not validation['is_valid']:
            # Fallback to safe response
            return "I apologize, but I cannot provide that response. How else can I help?"

        # Add disclaimers if needed
        if validation['needs_disclaimer']:
            response = self.validator.add_disclaimer(response, response)

        return response

    def chat(self, user_input: str) -> str:
        '''Safe chat with full pipeline.'''
        # Step 1: Filter input
        filtered_input = self._process_input(user_input)

        if filtered_input is None:
            return "I cannot process that input. Please rephrase."

        # Step 2: Generate response
        response = self._generate_response(filtered_input)

        # Step 3: Validate and process output
        safe_response = self._process_output(response)

        return safe_response

safe_twin = SafeDigitalTwin('Safe Assistant')

print('\nSafe Digital Twin:')
print(safe_twin.chat('Hello, how are you?'))
print(safe_twin.chat('Tell me about medical treatments'))  # Will add disclaimer


Safe Digital Twin:
I received: Hello, how are you?
I received: Tell me about medical treatments

⚠️ Note: This is not medical advice. Consult a healthcare professional.


## Exercise 1: Add More Guardrails

**Task**: Enhance the safety system:

1. Add more PII patterns (addresses, dates of birth)
2. Create custom content filters for your domain
3. Add rate limiting
4. Implement logging of safety violations
5. Test with edge cases

In [ ]:
# YOUR CODE HERE

# Enhance PIIDetector
# Add new patterns

# Create custom filters

# Test your improvements

In [20]:
import re
from typing import Dict, Any

class EnhancedPIIDetector(PIIDetector):
    '''Extends the base PIIDetector with date of birth (DOB) detection patterns.'''

    def __init__(self):
        super().__init__()
        # Add a regex pattern that matches:
        # - Numeric dates: MM/DD/YYYY, MM-DD-YYYY, YYYY-MM-DD, MM.DD.YYYY
        # - Textual dates: January 15, 1990, Oct 5, 1985
        self.patterns['dob'] = (
            r'\b(?:\d{1,2}[-/.]\d{1,2}[-/.]\d{4}|'  # Match numeric formats like 12/31/1999 or 1999-12-31
            r'\d{4}[-/.]\d{1,2}[-/.]\d{1,2}|'
            r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'
            r'Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+'
            r'\d{1,2},?\s+\d{4})\b' # Match textual formats like October 15, 1995
        )

# Instantiate and test the enhanced detector
enhanced_detector = EnhancedPIIDetector()
test_text = "Patient Jane Doe, born on 11/24/1993. Another record shows DOB as March 5, 1988."

print("Enhanced PII Detection:")
print("Original:", test_text)
print("Detection:", enhanced_detector.detect(test_text))
print("Redacted:", enhanced_detector.redact(test_text))

Enhanced PII Detection:
Original: Patient Jane Doe, born on 11/24/1993. Another record shows DOB as March 5, 1988.
Detection: {'has_pii': True, 'pii_types': ['dob'], 'details': {'dob': ['11/24/1993', 'March 5, 1988']}}
Redacted: Patient Jane Doe, born on [REDACTED_DOB]. Another record shows DOB as [REDACTED_DOB].


In [21]:
import time

class RateLimiter:
    '''A simple sliding window rate limiter.'''
    def __init__(self, max_calls: int, period: float):
        self.max_calls = max_calls
        self.period = period
        self.calls = []

    def allow_request(self) -> bool:
        current_time = time.time()
        # Filter out calls outside the sliding window
        self.calls = [t for t in self.calls if current_time - t < self.period]

        if len(self.calls) < self.max_calls:
            self.calls.append(current_time)
            return True
        return False

class RateLimitedPIIDetector(EnhancedPIIDetector):
    '''Extends the EnhancedPIIDetector with rate limiting capabilities.'''
    def __init__(self, max_calls: int = 3, period: float = 5.0):
        super().__init__()
        self.limiter = RateLimiter(max_calls, period)

    def detect(self, text: str) -> Dict[str, Any]:
        if not self.limiter.allow_request():
            return {
                'has_pii': False,
                'pii_types': [],
                'details': {},
                'error': 'Rate limit exceeded. Please try again later.'
            }
        return super().detect(text)

# Test Rate Limiting
limited_detector = RateLimitedPIIDetector(max_calls=2, period=4.0)

for i in range(4):
    res = limited_detector.detect("Test check from john@example.com")
    if 'error' in res:
        print(f"Call {i+1}: {res['error']}")
    else:
        print(f"Call {i+1}: Successfully processed!")
    time.sleep(0.5)

Call 1: Successfully processed!
Call 2: Successfully processed!
Call 3: Rate limit exceeded. Please try again later.
Call 4: Rate limit exceeded. Please try again later.


## Summary

### What We Learned:
- Content filtering blocks inappropriate content
- PII detection protects user privacy
- Output validation ensures quality
- Multiple safety layers = robust system
- Always add disclaimers for sensitive topics

### Next: Week 3.4 - Red Teaming & Security